# Aula 01 — Agente ReAct com LangGraph e Gemini

Este notebook implementa um agente de inventário usando o padrão **ReAct** (Reasoning + Acting),
construído do zero em três etapas evolutivas:

- **Etapa 1 — Loop manual:** `run_react_agent` interpreta respostas do LLM com regex, extrai ações
  e chama funções Python simulando ferramentas de consulta de estoque e preços.
- **Etapa 2 — Ferramentas aprimoradas:** refinamento do parsing de ações (`PAUSA`, regex mais robusta),
  nova ferramenta `encontrar_produto_mais_caro` e inspeção do histórico da conversa.
- **Etapa 3 — Expansão do toolkit:** nova ferramenta `calcular_valor_total_lista` e atualização
  do prompt com exemplos adicionais de uso.

> **Arquitetura:** diferentemente da Aula 04, este notebook **não usa o `StateGraph` do LangGraph**
> para orquestrar o ciclo — o loop ReAct é implementado manualmente via `for` + regex.
> O `StateGraph`/`TypedDict` importados são reservados para aulas futuras.


## 1. Instalação das Dependências

Pacotes necessários para o notebook:

| Pacote | Papel |
|--------|-------|
| `google-generativeai` | SDK Python para o Gemini (geração de texto, chat multi-turn) |
| `google-ai-generativelanguage==0.6.15` | Versão pinada da lib de proto/gRPC do Gemini |
| `langchain-google-genai` | Integração LangChain com o Gemini (usada nas aulas seguintes) |
| `langchain-community` | Ferramentas e utilitários da comunidade LangChain |
| `langgraph` | Framework de grafos de agentes (usado nas aulas seguintes) |


In [ ]:
%pip install -U google-generativeai
%pip install google-ai-generativelanguage==0.6.15
%pip install -U langchain-google-genai
%pip install -U langchain-community
%pip install -U langgraph


## 2. Imports

Importamos apenas o necessário para a implementação manual do loop ReAct:
- `re` para extração de ações via expressões regulares
- `google.generativeai` para o chat multi-turn com o Gemini
- `StateGraph`, `TypedDict` são importados aqui para uso nas etapas futuras


In [ ]:
import os 
import re  # Usado para extrair 'Ação:' e 'Resposta:' do texto do LLM
import google.generativeai as genai  # SDK oficial do Google para o Gemini
from langgraph.graph import StateGraph, END  # Reservados para aulas futuras
from typing import TypedDict

## 3. Configuração de Variáveis de Ambiente

As chaves de API são carregadas de um arquivo `.env` local (nunca hardcoded no código).
- `GEMINI_API_KEY` → mapeada para `GOOGLE_API_KEY`, nome esperado pelo SDK do Google
- `TAVILY_API_KEY` → carregada aqui para uso nas aulas posteriores


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ['GOOGLE_API_KEY'] = os.getenv('GEMINI_API_KEY') 
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

## 4. Verificação do Modelo

Validamos que a chave de API está configurada corretamente com um "Hello World"
antes de construir o agente completo. `genai.configure()` registra a chave globalmente
para todas as chamadas subsequentes do SDK.


In [ ]:
import os
from dotenv import load_dotenv
import google.generativeai as genai 

load_dotenv()

os.environ['GOOGLE_API_KEY'] = os.getenv('GEMINI_API_KEY')
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')

# Registra a chave globalmente no SDK — obrigatório antes de qualquer chamada genai.*
genai.configure(api_key=os.getenv('GOOGLE_API_KEY'))

# Teste rápido: garante que a API key é válida antes de implementar o agente
model = genai.GenerativeModel('gemini-2.5-flash')
response = model.generate_content("Hello world")

print(response.text)

## 5. Classe Agent — Versão Exploratória

> **Nota:** esta célula contém uma implementação exploratória que usa um `client` de estilo OpenAI
> (não definido neste notebook). Ela serve como rascunho conceitual do padrão:
> *acumular histórico de mensagens → chamar o modelo → guardar a resposta*.
> A implementação funcional está na função `run_react_agent` definida adiante.


In [ ]:
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []  # Histórico acumulado no formato [{role, content}]
        if self.system:
            # O system prompt é inserido como primeira mensagem no histórico
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        # Nota: 'client' referencia uma instância OpenAI — não definida aqui.
        # Esta versão é exploratória; o agente funcional usa genai diretamente.
        completion = client.chat.completions.create(
                            model="'gemini-2.5-flash'",
                            temperature=0,
                            messages=self.messages)
        return completion.choices[0].message.content

## 6. O Padrão ReAct e o Prompt de Sistema

**ReAct** (Reasoning + Acting) é um padrão em que o LLM intercala raciocínio com ações:

```
Pergunta → Pensamento → Ação → PAUSA
               ↑                 |
               └── Observação ←──┘
                       |
                   Resposta
```

O `PROMPT_REACT` instrui o modelo a:
1. **Pensar** sobre o que precisa descobrir
2. **Agir** chamando uma ferramenta com sintaxe `Ação: nome_ferramenta: argumento`
3. **Pausar** (retornar `PAUSA`) para que o código Python execute a ferramenta
4. Processar a **Observação** (resultado da ferramenta) e continuar o ciclo
5. Fornecer a **Resposta** final quando tiver informação suficiente

Nesta primeira versão, o agente tem acesso a duas ferramentas:
- `consultar_estoque` — quantidade disponível de um item
- `consultar_preco_produto` — preço unitário de um produto


In [ ]:
PROMPT_REACT = """
Você funciona em um ciclo de Pensamento, Ação, Pausa e Observação.
Ao final do ciclo, você fornece uma Resposta.
Use "Pensamento" para descrever seu raciocínio.
Use "Ação" para executar ferramentas - e então retorne "PAUSA".
A "Observação" será o resultado da ação executada.
Ações disponíveis:
  - consultar_estoque: retorna a quantidade disponível de um item no inventário (ex: "consultar_estoque: teclado")
  - consultar_preco_produto: retorna o preço unitário de um produto (ex: "consultar_preco_produto: mouse gamer")

Exemplo:
Pergunta: Quantos monitores temos em estoque?
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de monitores.
Ação: consultar_estoque: monitor
PAUSA

Observação: Temos 75 monitores em estoque.
Resposta: Há 75 monitores em estoque.
""".strip()


## 7. Estado do Agente (TypedDict)

`EstadoAgente` define o schema de estado para uso futuro com o `StateGraph` do LangGraph.
Neste notebook, o loop ReAct é implementado manualmente (sem `StateGraph`), mas a
definição aqui serve como documentação da estrutura que um grafo formal usaria:

| Campo | Tipo | Papel |
|-------|------|-------|
| `pergunta` | `str` | A pergunta original do usuário |
| `historico` | `list[str]` | Acúmulo do ciclo Pensamento/Ação/Observação |
| `acao_pendente` | `str` | Ação extraída aguardando execução |
| `resposta_final` | `str` | Resposta produzida pelo agente |


In [ ]:
class EstadoAgente(TypedDict): 
    pergunta: str           # A pergunta original do usuário
    historico: list[str]    # Acúmulo do ciclo: Pensamento, Ação, Observação
    acao_pendente: str      # Ação extraída do LLM, aguardando execução Python
    resposta_final: str     # Resposta final produzida pelo agente


## 8. Ferramentas de Inventário — Versão 1

As ferramentas são funções Python simples que **simulam** consultas a um sistema de estoque.
Em produção, estas funções chamariam APIs reais ou bancos de dados.

O ciclo ReAct funciona porque o código Python intercepta o texto `Ação: nome: arg` gerado
pelo LLM, executa a função correspondente e devolve o resultado como `Observação:` na
próxima mensagem do chat — fechando o ciclo de raciocínio.


In [ ]:
def consultar_estoque(item: str) -> str:
    """
    Simula a consulta de estoque de um item no inventário.
    """
    item = item.lower()
    estoque = {
        "monitor": 75,
        "teclado": 120,
        "mouse gamer": 80,
        "webcam": 40,
        "headset": 60,
        "impressora": 15
    }

    if item in estoque:
        return f"Temos {estoque[item]} {item}s em estoque."
    else:
        return f"Item '{item}' não encontrado no inventário."

def consultar_preco_produto(produto: str) -> str:
    """
    Simula a consulta do preço unitário de um produto.
    """
    produto = produto.lower()
    precos = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impressora": 750.00
    }

    if produto in precos:
        return f"O preço de um(a) {produto} é R$ {precos[produto]:.2f}."
    else:
        return f"Produto '{produto}' não encontrado na lista de preços."



### Verificação das Ferramentas

Testamos as funções diretamente antes de integrá-las ao loop ReAct, garantindo que
retornam strings no formato esperado pelo prompt (`Observação: ...`).


In [ ]:
print(consultar_estoque("teclado"))
print(consultar_preco_produto("impressora"))
print(consultar_estoque("monitor"))

## 9. Loop ReAct — Versão 1

A função `run_react_agent` implementa o ciclo ReAct **manualmente**:

```
pergunta
   │
   ▼
chat.send_message(current_prompt)
   │
   ├─ começa com 'Resposta:' → retorna resposta final
   │
   └─ contém 'Ação: nome: arg' → executa ferramenta
          │
          └─ current_prompt = 'Observação: resultado\nResposta:'
                 │
                 └─ volta ao topo (próxima iteração)
```

**Limitações desta versão:**
- Regex `r"Ação:\s*(\w+):\s*(.*)"` não captura ações sem argumentos (como `encontrar_produto_mais_caro`)
- O `current_prompt` injeta `\nResposta:` para forçar a resposta na mesma mensagem — isso
  pode confundir o modelo em alguns casos
- `max_iterations` evita loops infinitos se o LLM não convergir


In [ ]:
def run_react_agent(pergunta: str, max_iterations: int = 5) -> str:
    model = genai.GenerativeModel('gemini-2.5-flash')

    # Inicia uma sessão de chat multi-turn; o histórico é mantido internamente pelo SDK
    chat = model.start_chat(history=[])

    # Envia o system prompt como primeira mensagem para definir o comportamento do agente
    chat.send_message(PROMPT_REACT)

    current_prompt = pergunta

    for i in range(max_iterations):
        response = chat.send_message(current_prompt)
        response_text = response.text.strip()

        print(f"--- Iteração {i+1} ---")
        print(f"Modelo pensou/respondeu:\n{response_text}\n")

        # Condição de parada: o LLM produziu uma resposta final
        if response_text.startswith("Resposta:"):
            return response_text.replace("Resposta:", "").strip()

        # Extrai 'Ação: nome_ferramenta: argumento' do texto do LLM
        # Limitação: não captura ações sem argumentos (ex: encontrar_produto_mais_caro)
        match = re.search(r"Ação:\s*(\w+):\s*(.*)", response_text)

        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip()

            observacao = ""
            if action_name == "consultar_estoque":
                observacao = consultar_estoque(action_arg)
            elif action_name == "consultar_preco_produto":
                observacao = consultar_preco_produto(action_arg)
            else:
                observacao = f"Erro: Ação '{action_name}' desconhecida."

            # Injeta o resultado como Observação e solicita a Resposta na mesma mensagem
            current_prompt = f"Observação: {observacao}\nResposta:"

            print(f"Executou ação: {action_name}({action_arg})")
            print(f"Observação: {observacao}\n")

        else:
            return f"Erro: O agente não conseguiu extrair uma Ação ou Resposta final após {i+1} iterações. Última resposta: {response_text}"

    return "Erro: Limite máximo de iterações atingido sem uma resposta final."


## 10. Demonstrações com o Agente v1

Testamos o agente com quatro perguntas que cobrem os casos de uso das ferramentas.
A **Interação 4** (`produto mais caro`) deve falhar nesta versão — a ferramenta
`encontrar_produto_mais_caro` ainda não existe, demonstrando a necessidade da Etapa 2.


In [ ]:
pergunta_1 = "Quantos mouses gamers estão no inventário?"
print(f"**Interação 1: {pergunta_1}**")
resposta_1 = run_react_agent(pergunta_1)
print(f"\n**RESPOSTA FINAL DO AGENTE 1:** {resposta_1}\n")

print("\n" + "="*50 + "\n") 


In [ ]:

pergunta_2 = "Qual o valor de uma impressora?"
print(f"**Interação 2: {pergunta_2}**")
resposta_2 = run_react_agent(pergunta_2)
print(f"\n**RESPOSTA FINAL DO AGENTE 2:** {resposta_2}\n")

print("\n" + "="*50 + "\n") 



In [ ]:
pergunta_3 = "Temos cadeiras em estoque?"
print(f"**Interação 3: {pergunta_3}**")
resposta_3 = run_react_agent(pergunta_3)
print(f"\n**RESPOSTA FINAL DO AGENTE 3:** {resposta_3}\n")

In [ ]:
pergunta_4 = "Qual é o produto mais caro?"
print(f"**Interação 4: {pergunta_4}**")
resposta_4 = run_react_agent(pergunta_4)
print(f"\n**RESPOSTA FINAL DO AGENTE 4:** {resposta_4}\n")

## 11. Evolução das Ferramentas — Versão 2

Esta etapa traz três melhorias em relação à versão anterior:

1. **`.strip()` nos argumentos:** evita erros de lookup por espaços extras que o LLM
   pode incluir nos argumentos (ex: `" monitor"` vs `"monitor"`).

2. **Nova ferramenta `encontrar_produto_mais_caro`:** usa `max()` com `key=dict.get`
   para encontrar o item de maior valor sem argumentos do LLM. O prompt precisará
   ser atualizado para ensiná-lo a chamá-la.

3. **Redefinição das funções existentes:** Python permite redefinir funções em células
   subsequentes; a nova definição sobrescreve silenciosamente a anterior no namespace.


In [ ]:
def consultar_estoque(item: str) -> str: 
    """
    Simula a consulta de estoque de um item no inventário.
    """
    # .strip() adicionado: o LLM pode gerar espaços extras nos argumentos
    item = item.lower().strip()
    estoque = {
        "monitor": 75,
        "teclado": 120,
        "mouse gamer": 80,
        "webcam": 40,
        "headset": 60,
        "impressora": 15
    }

    if item in estoque:
        return f"Temos {estoque[item]} {item}s em estoque."
    else:
        return f"Item '{item}' não encontrado no inventário."

def consultar_preco_produto(produto: str) -> str: 
    """
    Simula a consulta do preço unitário de um produto.
    """
    # .strip() adicionado: normaliza o argumento antes do lookup
    produto = produto.lower().strip()
    precos = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impressora": 750.00
    }

    if produto in precos:
        return f"O preço de um(a) {produto} é R$ {precos[produto]:.2f}."
    else:
        return f"Produto '{produto}' não encontrado na lista de preços."

def ferramenta_encontrar_produto_mais_caro() -> str: 
    """
    Retorna o nome e o preço do produto mais caro no inventário.
    Esta função não precisa de argumentos.
    """
    # Dicionário local espelhando os preços das ferramentas acima
    precos_do_inventario = {
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impressora": 750.00
    }

    if not precos_do_inventario: 
        return "Nenhum produto encontrado na lista de preços para comparação."

    # max() com key=dict.get retorna a CHAVE (nome do produto) com o maior VALOR (preço)
    nome_produto_mais_caro = max(precos_do_inventario, key=precos_do_inventario.get)
    valor_produto_mais_caro = precos_do_inventario[nome_produto_mais_caro]

    return f"O produto mais caro é o(a) {nome_produto_mais_caro} com preço de R$ {valor_produto_mais_caro:.2f}." 


### Prompt ReAct v2 — Adicionando `encontrar_produto_mais_caro`

O prompt é atualizado para incluir a nova ferramenta e um **exemplo de uso** dela.
Exemplos de few-shot no prompt são essenciais para ensinar ao LLM a sintaxe correta
de chamada de ferramentas sem argumentos (`Ação: encontrar_produto_mais_caro` — sem `:`
após o nome).


In [ ]:
PROMPT_REACT = """
Você funciona em um ciclo de Pensamento, Ação, Pausa e Observação.
Ao final do ciclo, você fornece uma Resposta.
Use "Pensamento" para descrever seu raciocínio.
Use "Ação" para executar ferramentas - e então retorne "PAUSA".
A "Observação" será o resultado da ação executada.
Ações disponíveis:
  - consultar_estoque: retorna a quantidade disponível de um item no inventário (ex: "consultar_estoque: teclado")
  - consultar_preco_produto: retorna o preço unitário de um produto (ex: "consultar_preco_produto: mouse gamer")
  - encontrar_produto_mais_caro: retorna o nome e o preço do produto mais caro no inventário (não requer argumentos)

Exemplo:
Pergunta: Quantos monitores temos em estoque?
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de monitores.
Ação: consultar_estoque: monitor
PAUSA

Observação: Temos 75 monitores em estoque.
Resposta: Há 75 monitores em estoque.

Exemplo:
Pergunta: Qual é o produto mais caro?
Pensamento: Preciso usar a ação encontrar_produto_mais_caro para descobrir qual produto tem o maior preço.
Ação: encontrar_produto_mais_caro
PAUSA

Observação: O produto mais caro é o(a) monitor com preço de R$ 999.90.
Resposta: O produto mais caro é o(a) monitor com preço de R$ 999.90.
""".strip()

## 12. Loop ReAct — Versão 2

Melhorias em relação à v1:

1. **Regex de resposta mais robusta:** `re.search(r"Resposta:\s*(.*)", ..., re.DOTALL)`
   captura respostas em qualquer posição do texto (não apenas no início) e multi-linha.

2. **Regex de ação com argumento opcional:** `r"Ação:\s*(\w+)(?::\s*([^\n]*))?"` —
   o grupo `(?::\s*...)?` torna o argumento opcional, permitindo chamar
   `encontrar_produto_mais_caro` sem argumentos.

3. **`Observação:` sem `\nResposta:`:** simplificação do prompt de Observação —
   o LLM decide naturalmente se já tem informação suficiente para responder.


In [ ]:
def run_react_agent(pergunta: str, max_iterations: int = 5) -> str:
    """
    Executa o ciclo ReAct para uma dada pergunta usando o modelo Gemini.
    """

    model = genai.GenerativeModel('gemini-2.5-flash')

    # start_chat(history=[]) cria uma nova sessão sem contexto anterior
    chat = model.start_chat(history=[])
    chat.send_message(PROMPT_REACT)

    current_prompt = pergunta

    for i in range(max_iterations):
        response = chat.send_message(current_prompt)
        response_text = response.text.strip()

        print(f"\n--- Iteração {i+1} ---")
        print(f"Modelo pensou/respondeu:\n{response_text}\n")

        # Melhoria v2: re.DOTALL permite capturar respostas multi-linha
        response_match_final = re.search(r"Resposta:\s*(.*)", response_text, re.DOTALL)
        if response_match_final:
            return response_match_final.group(1).strip()

        # Melhoria v2: argumento opcional com (?::\s*([^\n]*))?  — captura ações sem args
        match = re.search(r"Ação:\s*(\w+)(?::\s*([^\n]*))?" , response_text)

        if match:
            action_name = match.group(1).strip()

            # group(2) é None quando não há argumento (ex: encontrar_produto_mais_caro)
            action_arg = match.group(2).strip() if match.group(2) is not None else ""

            observacao_da_acao = ""

            if action_name == "consultar_estoque":
                observacao_da_acao = consultar_estoque(action_arg)
            elif action_name == "consultar_preco_produto":
                observacao_da_acao = consultar_preco_produto(action_arg)
            elif action_name == "encontrar_produto_mais_caro": 
                # Ferramenta sem argumentos: não passa action_arg
                observacao_da_acao = ferramenta_encontrar_produto_mais_caro()
            else:
                observacao_da_acao = f"Erro: Ação '{action_name}' desconhecida. Verifique o prompt ou a implementação da ferramenta."

            # Melhoria v2: Observação sem '\nResposta:' — LLM decide quando encerrar
            current_prompt = f"Observação: {observacao_da_acao}"

            print(f"Executou ação: {action_name} com argumento '{action_arg}'")
            print(f"Observação: {observacao_da_acao}\n")

        else:
            return f"Erro: O agente não conseguiu extrair uma Ação ou Resposta final após {i+1} iterações. Última resposta do modelo: {response_text}"

    return "Erro: Limite máximo de iterações atingido sem uma resposta final do agente."


In [ ]:
print("--- Começando as Interações com o Agente ReAct ---")

# Interação 1: Consultar Estoque
pergunta_1 = "Quantos teclados temos em estoque?"
print(f"\n**Interação 1: {pergunta_1}**")
resposta_1 = run_react_agent(pergunta_1)
print(f"\n**RESPOSTA FINAL DO AGENTE 1:** {resposta_1}\n")

print("\n" + "="*80 + "\n")

# Interação 2: Consultar Preço
pergunta_2 = "Qual o preço de um headset?"
print(f"\n**Interação 2: {pergunta_2}**")
resposta_2 = run_react_agent(pergunta_2)
print(f"\n**RESPOSTA FINAL DO AGENTE 2:** {resposta_2}\n")

print("\n" + "="*80 + "\n")

# Interação 3: Item não encontrado no estoque
pergunta_3 = "Temos cadeiras em estoque?"
print(f"\n**Interação 3: {pergunta_3}**")
resposta_3 = run_react_agent(pergunta_3)
print(f"\n**RESPOSTA FINAL DO AGENTE 3:** {resposta_3}\n")

print("\n" + "="*80 + "\n")

# Interação 4: Encontrar o Produto Mais Caro (A nova funcionalidade!)
pergunta_4 = "Qual é o produto mais caro?"
print(f"\n**Interação 4: {pergunta_4}**")
resposta_4 = run_react_agent(pergunta_4)
print(f"\n**RESPOSTA FINAL DO AGENTE 4:** {resposta_4}\n")

print("\n--- Fim das Interações ---")

## 13. Inspecionando o Histórico da Conversa

`run_react_agent_with_history` é uma variante que retorna também o `chat.history`
do SDK do Google — a lista de todas as mensagens trocadas com o modelo.

Isso permite **inspecionar o raciocínio completo** do agente:
- As mensagens com `role: user` contêm o prompt inicial + as `Observação:` enviadas
- As mensagens com `role: model` contêm o raciocínio ReAct (`Pensamento:`, `Ação:`, `PAUSA`, `Resposta:`)

Esta funcionalidade é útil para debug, para entender o comportamento do agente e
para implementar persistência manual de histórico entre sessões.


In [ ]:
def run_react_agent_with_history(pergunta: str, max_iterations: int = 5) -> tuple[str, list]: 
    model = genai.GenerativeModel('gemini-2.5-flash') 

    chat = model.start_chat(history=[])
    # O PROMPT_REACT é enviado como primeira mensagem para configurar o comportamento
    chat.send_message(PROMPT_REACT) 

    current_prompt = pergunta

    for i in range(max_iterations):
        response = chat.send_message(current_prompt)
        response_text = response.text.strip()

        print(f"\n--- Iteração {i+1} ---")
        print(f"Modelo pensou/respondeu:\n{response_text}\n")

        response_match_final = re.search(r"Resposta:\s*(.*)", response_text, re.DOTALL)
        if response_match_final:
            # Retorna a resposta E o histórico completo para inspeção
            return response_match_final.group(1).strip(), chat.history

        match = re.search(r"Ação:\s*(\w+)(?::\s*([^\n]*))?" , response_text)

        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip() if match.group(2) is not None else ""

            observacao_da_acao = ""

            if action_name == "consultar_estoque":
                observacao_da_acao = consultar_estoque(action_arg)
            elif action_name == "consultar_preco_produto":
                observacao_da_acao = consultar_preco_produto(action_arg)
            elif action_name == "encontrar_produto_mais_caro":
                observacao_da_acao = ferramenta_encontrar_produto_mais_caro()
            else:
                observacao_da_acao = f"Erro: Ação '{action_name}' desconhecida. Verifique o prompt ou a implementação da ferramenta."

            current_prompt = f"Observação: {observacao_da_acao}"

            print(f"Executou ação: {action_name} com argumento '{action_arg}'")
            print(f"Observação: {observacao_da_acao}\n")

        else:
            return f"Erro: O agente não conseguiu extrair uma Ação ou Resposta final após {i+1} iterações. Última resposta do modelo: {response_text}", chat.history

    return "Erro: Limite máximo de iterações atingido sem uma resposta final do agente.", chat.history


In [ ]:
pergunta_exemplo = "Quantos teclados temos em estoque?"
resposta_exemplo, historico_completo = run_react_agent_with_history(pergunta_exemplo)

print(f"**RESPOSTA FINAL DO AGENTE:** {resposta_exemplo}\n")

print("\n--- Histórico Completo da Interação ---")
for i, message in enumerate(historico_completo):
    print(f"--- Mensagem {i+1} (role: {message.role}) ---")

    for part in message.parts:
        if hasattr(part, 'text'):
            print(part.text)  # Texto da mensagem (Pensamento, Ação, Observação, etc.)
        else:
            print(part)  # Outros tipos de conteúdo (imagens, blobs, etc.)
    print("-" * 20)
print("\n--- Fim do Histórico ---\n")


## 14. Nova Ferramenta — Cálculo de Valor Total de Lista

`ferramenta_calcular_valor_total_lista` recebe uma **string de itens separados por vírgula**
(ex: `"teclado, mouse gamer, monitor"`) e retorna o valor total dos que foram encontrados.

Esta ferramenta demonstra dois pontos importantes:
1. **Argumento composto:** o LLM precisa aprender a passar múltiplos itens em uma única
   string — isso exige um exemplo de few-shot específico no prompt.
2. **Resultado parcial:** itens não encontrados são listados separadamente, dando
   transparência ao LLM sobre o que foi incluído no cálculo.


In [ ]:
def ferramenta_calcular_valor_total_lista(lista_itens: str) -> str:
    """
    Calcula o valor total de uma lista de itens de compra.
    Recebe uma string com itens separados por vírgula (ex: "teclado, mouse gamer, monitor").
    """
    precos_do_inventario = { 
        "monitor": 999.90,
        "teclado": 150.00,
        "mouse gamer": 99.50,
        "webcam": 120.00,
        "headset": 180.00,
        "impressora": 750.00
    }

    # Normaliza cada item: remove espaços e converte para minúsculas
    itens_processados = [item.strip().lower() for item in lista_itens.split(',')]

    valor_total = 0.0
    itens_nao_encontrados = []

    for item in itens_processados: 
        if item in precos_do_inventario:
            valor_total += precos_do_inventario[item]
        else:
            # Acumula itens ausentes para incluir no retorno — transparência para o LLM
            itens_nao_encontrados.append(item)

    resposta = f"O valor total dos itens encontrados é R$ {valor_total:.2f}."
    if itens_nao_encontrados:
        # Informa ao LLM quais itens ficaram de fora do cálculo
        resposta += f" Os seguintes itens não foram encontrados e não foram incluídos no cálculo: {', '.join(itens_nao_encontrados)}."

    return resposta


### Verificação da Nova Ferramenta — Três Cenários

Cobrimos três cenários para garantir a robustez:
1. **Todos os itens encontrados** — total calculado corretamente
2. **Alguns itens ausentes** — total parcial + lista dos não encontrados
3. **Nenhum item encontrado** — total zero + todos listados como ausentes


In [ ]:
print("Testando ferramenta_calcular_valor_total_lista:")

lista_1 = "teclado, mouse gamer, monitor"
resultado_1 = ferramenta_calcular_valor_total_lista(lista_1)
print(f"Lista: '{lista_1}'\nResultado: {resultado_1}\n")


In [ ]:
lista_2 = "headset, cadeira"
resultado_2 = ferramenta_calcular_valor_total_lista(lista_2)
print(f"Lista: '{lista_2}'\nResultado: {resultado_2}\n")


In [ ]:
lista_3 = "mesa, copo"
resultado_3 = ferramenta_calcular_valor_total_lista(lista_3)
print(f"Lista: '{lista_3}'\nResultado: {resultado_3}\n")


## 15. Loop ReAct — Versão 3

A v3 adiciona suporte à ferramenta `calcular_valor_total_lista` no dispatcher do loop.
A estrutura do `if/elif` é idêntica às versões anteriores — cada nova ferramenta
requer apenas um novo `elif` aqui e um exemplo no `PROMPT_REACT`.

> **Padrão de extensão:** para adicionar uma nova ferramenta ao agente:
> 1. Implementar a função Python
> 2. Adicionar o `elif action_name == "nome"` neste dispatcher
> 3. Atualizar o `PROMPT_REACT` com a descrição e um exemplo de uso


In [ ]:
def run_react_agent(pergunta: str, max_iterations: int = 5) -> str:
    model = genai.GenerativeModel('gemini-2.5-flash')

    chat = model.start_chat(history=[])
    chat.send_message(PROMPT_REACT)

    current_prompt = pergunta

    for i in range(max_iterations):
        response = chat.send_message(current_prompt)
        response_text = response.text.strip()

        print(f"\n--- Iteração {i+1} ---")
        print(f"Modelo pensou/respondeu:\n{response_text}\n")

        response_match_final = re.search(r"Resposta:\s*(.*)", response_text, re.DOTALL)
        if response_match_final:
            return response_match_final.group(1).strip()

        match = re.search(r"Ação:\s*(\w+)(?::\s*([^\n]*))?" , response_text)

        if match:
            action_name = match.group(1).strip()
            action_arg = match.group(2).strip() if match.group(2) is not None else ""

            observacao_da_acao = ""

            if action_name == "consultar_estoque":
                observacao_da_acao = consultar_estoque(action_arg)
            elif action_name == "consultar_preco_produto":
                observacao_da_acao = consultar_preco_produto(action_arg)
            elif action_name == "encontrar_produto_mais_caro":
                observacao_da_acao = ferramenta_encontrar_produto_mais_caro()
            elif action_name == "calcular_valor_total_lista":  # Nova ferramenta v3
                observacao_da_acao = ferramenta_calcular_valor_total_lista(action_arg)
            else:
                observacao_da_acao = f"Erro: Ação '{action_name}' desconhecida. Verifique o prompt ou a implementação da ferramenta."

            current_prompt = f"Observação: {observacao_da_acao}"

            print(f"Executou ação: {action_name} com argumento '{action_arg}'")
            print(f"Observação: {observacao_da_acao}\n")

        else:
            return f"Erro: O agente não conseguiu extrair uma Ação ou Resposta final após {i+1} iterações. Última resposta do modelo: {response_text}"

    return "Erro: Limite máximo de iterações atingido sem uma resposta final do agente."


### Demonstração v3 — Prompt ainda sem exemplo de `calcular_valor_total_lista`

Esta rodada usa o loop v3 (que já suporta a nova ferramenta) mas com o `PROMPT_REACT`
da v2 (sem o exemplo de `calcular_valor_total_lista`). A Interação 5 pode falhar
ou produzir resultados inesperados — demonstrando que **o loop e o prompt precisam
evoluir juntos**.


In [ ]:
print("--- Começando as Interações com o Agente ReAct ---")

# Interação 1: Consultar Estoque
pergunta_1 = "Quantos teclados temos em estoque?"
print(f"\n**Interação 1: {pergunta_1}**")
resposta_1 = run_react_agent(pergunta_1)
print(f"\n**RESPOSTA FINAL DO AGENTE 1:** {resposta_1}\n")

print("\n" + "="*80 + "\n")

# Interação 2: Consultar Preço
pergunta_2 = "Qual o preço de um headset?"
print(f"\n**Interação 2: {pergunta_2}**")
resposta_2 = run_react_agent(pergunta_2)
print(f"\n**RESPOSTA FINAL DO AGENTE 2:** {resposta_2}\n")

print("\n" + "="*80 + "\n")

# Interação 3: Item não encontrado no estoque
pergunta_3 = "Temos cadeiras em estoque?"
print(f"\n**Interação 3: {pergunta_3}**")
resposta_3 = run_react_agent(pergunta_3)
print(f"\n**RESPOSTA FINAL DO AGENTE 3:** {resposta_3}\n")

print("\n" + "="*80 + "\n")

# Interação 4: Encontrar o Produto Mais Caro
pergunta_4 = "Qual é o produto mais caro?"
print(f"\n**Interação 4: {pergunta_4}**")
resposta_4 = run_react_agent(pergunta_4)
print(f"\n**RESPOSTA FINAL DO AGENTE 4:** {resposta_4}\n")

print("\n" + "="*80 + "\n")

# Interação 5: Calcular Valor Total da Lista (NOVA FUNCIONALIDADE)
pergunta_5 = "Qual o valor de um teclado, uma impressora e uma webcam?"
print(f"\n**Interação 5: {pergunta_5}**")
resposta_5 = run_react_agent(pergunta_5)
print(f"\n**RESPOSTA FINAL DO AGENTE 5:** {resposta_5}\n")


print("\n--- Fim das Interações ---")

### Prompt ReAct v3 — Adicionando `calcular_valor_total_lista`

O prompt é atualizado com a descrição da nova ferramenta e um exemplo de few-shot
mostrando como passar múltiplos itens como argumento: `calcular_valor_total_lista: teclado, mouse gamer`.

O exemplo é crucial: sem ele, o LLM pode tentar chamar `consultar_preco_produto`
múltiplas vezes para cada item em vez de usar a nova ferramenta agregada.


In [ ]:
PROMPT_REACT = """
Você funciona em um ciclo de Pensamento, Ação, Pausa e Observação.
Ao final do ciclo, você fornece uma Resposta.
Use "Pensamento" para descrever seu raciocínio.
Use "Ação" para executar ferramentas - e então retorne "PAUSA".
A "Observação" será o resultado da ação executada.
Ações disponíveis:
  - consultar_estoque: retorna a quantidade disponível de um item no inventário (ex: "consultar_estoque: teclado")
  - consultar_preco_produto: retorna o preço unitário de um produto (ex: "consultar_preco_produto: mouse gamer")
  - encontrar_produto_mais_caro: retorna o nome e o preço do produto mais caro no inventário (não requer argumentos)
  - calcular_valor_total_lista: calcula o valor total de uma lista de itens de compra. Recebe uma string com itens separados por vírgula (ex: "teclado, mouse gamer, monitor")

Exemplo:
Pergunta: Quantos monitores temos em estoque?
Pensamento: Devo consultar a ação consultar_estoque para saber a quantidade de monitores.
Ação: consultar_estoque: monitor
PAUSA

Observação: Temos 75 monitores em estoque.
Resposta: Há 75 monitores em estoque.

Exemplo:
Pergunta: Qual é o produto mais caro?
Pensamento: Preciso usar a ação encontrar_produto_mais_caro para descobrir qual produto tem o maior preço.
Ação: encontrar_produto_mais_caro
PAUSA

Observação: O produto mais caro é o(a) monitor com preço de R$ 999.90.
Resposta: O produto mais caro é o(a) monitor com preço de R$ 999.90.

Exemplo:
Pergunta: Quanto custa um teclado e um mouse gamer?
Pensamento: O usuário quer saber o valor total de vários itens. Devo usar a ação calcular_valor_total_lista com os itens "teclado, mouse gamer".
Ação: calcular_valor_total_lista: teclado, mouse gamer
PAUSA

Observação: O valor total dos itens encontrados é R$ 249.50.
Resposta: O valor total do teclado e do mouse gamer é R$ 249.50.
""".strip()


### Demonstração Final — Loop v3 + Prompt v3

Com prompt e loop alinhados, todas as cinco interações devem funcionar corretamente:

| Interação | Pergunta | Ferramenta esperada |
|-----------|----------|---------------------|
| 1 | Quantos teclados? | `consultar_estoque` |
| 2 | Preço do headset? | `consultar_preco_produto` |
| 3 | Temos cadeiras? | `consultar_estoque` (não encontrado) |
| 4 | Produto mais caro? | `encontrar_produto_mais_caro` |
| 5 | Valor de teclado + impressora + webcam? | `calcular_valor_total_lista` |


In [ ]:
print("--- Começando as Interações com o Agente ReAct ---")

# Interação 1: Consultar Estoque
pergunta_1 = "Quantos teclados temos em estoque?"
print(f"\n**Interação 1: {pergunta_1}**")
resposta_1 = run_react_agent(pergunta_1)
print(f"\n**RESPOSTA FINAL DO AGENTE 1:** {resposta_1}\n")

print("\n" + "="*80 + "\n")

# Interação 2: Consultar Preço
pergunta_2 = "Qual o preço de um headset?"
print(f"\n**Interação 2: {pergunta_2}**")
resposta_2 = run_react_agent(pergunta_2)
print(f"\n**RESPOSTA FINAL DO AGENTE 2:** {resposta_2}\n")

print("\n" + "="*80 + "\n")

# Interação 3: Item não encontrado no estoque
pergunta_3 = "Temos cadeiras em estoque?"
print(f"\n**Interação 3: {pergunta_3}**")
resposta_3 = run_react_agent(pergunta_3)
print(f"\n**RESPOSTA FINAL DO AGENTE 3:** {resposta_3}\n")

print("\n" + "="*80 + "\n")

# Interação 4: Encontrar o Produto Mais Caro
pergunta_4 = "Qual é o produto mais caro?"
print(f"\n**Interação 4: {pergunta_4}**")
resposta_4 = run_react_agent(pergunta_4)
print(f"\n**RESPOSTA FINAL DO AGENTE 4:** {resposta_4}\n")

print("\n" + "="*80 + "\n")

# Interação 5: Calcular Valor Total da Lista (NOVA FUNCIONALIDADE)
pergunta_5 = "Qual o valor de um teclado, uma impressora e uma webcam?"
print(f"\n**Interação 5: {pergunta_5}**")
resposta_5 = run_react_agent(pergunta_5)
print(f"\n**RESPOSTA FINAL DO AGENTE 5:** {resposta_5}\n")


print("\n--- Fim das Interações ---")

## 16. Interface Interativa

`iniciar_conversacao_com_agente` cria um loop de entrada do usuário no terminal.
Cada pergunta recebe uma nova sessão `run_react_agent` — sem memória entre perguntas.

> **Nota sobre o ambiente Jupyter:** `input()` funciona em notebooks locais, mas pode
> não funcionar em ambientes remotos (Google Colab, JupyterHub). Neste caso, use
> diretamente `run_react_agent("sua pergunta")` nas células anteriores.

O bloco `if __name__ == "__main__"` garante que a função só é chamada automaticamente
quando o script é executado diretamente — não quando importado como módulo.


In [ ]:
def iniciar_conversacao_com_agente():
    print("--- Agente de Inventário Interativo ---")
    print("Digite sua pergunta sobre o inventário, ou digite 'sair' para encerrar.")
    print("-" * 50)

    while True:
        pergunta_usuario = input("\nVocê: ")

        if pergunta_usuario.lower().strip() == 'sair':
            print("Encerrando a conversa. Até logo!")
            break

        print("\nAgente: Processando...")
        try:
            # Cada pergunta cria uma sessão nova — sem memória entre rodadas
            resposta_agente = run_react_agent(pergunta_usuario)
            print(f"\nAgente: {resposta_agente}")
        except Exception as e:
            # Captura erros de API (rate limit, timeout, etc.) sem encerrar o loop
            print(f"\nAgente: Ocorreu um erro ao processar sua pergunta: {e}")
            print("Por favor, tente novamente ou digite 'sair'.")

if __name__ == "__main__":
    iniciar_conversacao_com_agente()
